# AIBackends - PDF redact -> extract / summarize / role-match pipelines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/AIBackends-PDF-redact-extract-workflows.ipynb)

Compose built-in workflow steps into privacy-first document pipelines: read a PDF, redact
PII locally with GLiNER, then hand only the redacted text to a local LLM. Covers
`examples/workflows/invoice_redact_extract.py`, `resume_redact_summarize.py`, and
`resume_role_match.py`.

**Runtime:** works on a CPU runtime; for faster inference pick *Runtime > Change runtime type > T4 GPU*. The device is detected automatically.

In [ ]:
import shutil
import subprocess

# Prebuilt llama-cpp-python wheels: CUDA 12.4 build on GPU runtimes, CPU build otherwise.
HAS_NVIDIA_GPU = (
    shutil.which("nvidia-smi") is not None
    and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
)
LLAMA_WHEEL = (
    "https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.35-cu124/llama_cpp_python-0.3.35-py3-none-manylinux_2_35_x86_64.whl"
    if HAS_NVIDIA_GPU
    else "https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.35/llama_cpp_python-0.3.35-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl"
)
print("GPU runtime:", HAS_NVIDIA_GPU)
print("llama-cpp-python wheel:", LLAMA_WHEEL.rsplit("/", 2)[-2])

%pip install -q "{LLAMA_WHEEL}"
%pip install -q "aibackends[pdf,pii]>=0.8.1" huggingface_hub

In [2]:
import shutil
import subprocess

import aibackends
import llama_cpp

# "gpu" offloads every layer to CUDA (n_gpu_layers=-1); "cpu" keeps everything on CPU.
HAS_NVIDIA_GPU = (
    shutil.which("nvidia-smi") is not None
    and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
)
DEVICE = "gpu" if HAS_NVIDIA_GPU and llama_cpp.llama_supports_gpu_offload() else "cpu"

print("aibackends", aibackends.__version__)
print("llama-cpp-python", llama_cpp.__version__)
print("device:", DEVICE)

aibackends 0.8.1
llama-cpp-python 0.3.35
device: cpu


In [3]:
from pathlib import Path
from urllib.request import urlretrieve

DATA_URL = "https://raw.githubusercontent.com/donvito/aibackends/main/examples/data"
DATA_DIR = Path("aibackends_data")


def fetch(relative_path: str) -> Path:
    """Download a sample file from the aibackends examples once and return its path."""
    path = DATA_DIR / relative_path
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        partial = path.with_name(path.name + ".part")
        try:
            urlretrieve(f"{DATA_URL}/{relative_path}", partial)
            partial.replace(path)
        finally:
            partial.unlink(missing_ok=True)
    return path

## Model

`GEMMA4_E2B` resolves to `unsloth/gemma-4-E2B-it-GGUF` (Q4_K_M) on the llama.cpp runtime and
is downloaded from the Hugging Face Hub on first use. Passing `device=DEVICE` offloads all
layers to CUDA on a GPU runtime and keeps inference on CPU otherwise.

In [4]:
from pydantic import BaseModel

from aibackends.models import GEMMA4_E2B
from aibackends.runtimes import LLAMACPP
from aibackends.steps.enrich import LLMAnalyser, LLMTextGenerator, PIIRedactor, TaskRunner
from aibackends.steps.ingest import FileIngestor, PDFIngestor
from aibackends.workflows import Pipeline

LLM = {"runtime": LLAMACPP, "model": GEMMA4_E2B, "device": DEVICE}


class PIISummary(BaseModel):
    backend: str
    entities_redacted: int
    redaction_map: dict[str, str]


def pii_summary(redaction) -> PIISummary:
    return PIISummary(
        backend=redaction.backend_used,
        entities_redacted=len(redaction.entities_found),
        redaction_map=redaction.redaction_map,
    )

## 1. Invoice PDF: extract text -> redact -> structured invoice JSON

In [5]:
from aibackends.schemas.invoice import InvoiceOutput


class InvoiceRedactExtractWorkflow(Pipeline):
    steps = [
        PDFIngestor(),
        PIIRedactor(
            backend="gliner",
            labels=["person_name", "email", "phone_number", "address", "account_number",
                    "tax_identifier"],
        ),
        LLMAnalyser(
            schema=InvoiceOutput,
            task_name="invoice-redacted-extract",
            system_prompt="You extract structured invoice data from redacted invoice text.",
            prompt=(
                "Extract the invoice from the redacted invoice text below. Use exactly these "
                "keys: vendor, line_items (each with description, quantity, unit_price, "
                "amount), subtotal, tax, total, due_date, payment_terms.\n"
                "quantity, unit_price, and amount must be plain numbers (e.g. 1, not "
                "\"1 mo.\"); put units in the description.\n"
                "Keep bracketed placeholders such as [PERSON_NAME_1] exactly as written.\n"
                "Do not guess redacted values.\n"
                "If a field is missing, return null for optional values."
            ),
            input_key="text",
            output_key="invoice",
        ),
    ]


raw = InvoiceRedactExtractWorkflow(**LLM).run(fetch("pdf/invoice.pdf"))
print(pii_summary(raw["pii_redaction"]).model_dump_json(indent=2))
print(raw["invoice"].model_dump_json(indent=2))

{
  "backend": "gliner",
  "entities_redacted": 9,
  "redaction_map": {
    "1540 Broadway, Suite 2200": "[ADDRESS_1]",
    "invoices@boldmediagroup.com": "[EMAIL_2]",
    "+1 (212) 555-0198": "[PHONE_NUMBER_3]",
    "Sarah Mitchell": "[PERSON_NAME_4]",
    "88 Commerce Blvd, Floor 4": "[ADDRESS_5]",
    "s.mitchell@pinnacleretail.com": "[EMAIL_6]",
    "Derek": "[PERSON_NAME_7]",
    "Okafor": "[PERSON_NAME_8]",
    "000012345678": "[ACCOUNT_NUMBER_9]"
  }
}
{
  "vendor": "Bold Media Group",
  "line_items": [
    {
      "description": "Programmatic Display Advertising (CPM campaign, 1.2M impressions, Q1 Spring push)",
      "quantity": 1.0,
      "unit_price": 1200000.0,
      "amount": 10200.0
    },
    {
      "description": "Social Media Sponsored Posts (Instagram + Facebook, 12 posts, boosted reach)",
      "quantity": 12.0,
      "unit_price": 650.0,
      "amount": 7800.0
    },
    {
      "description": "Google Search Ads Management (Keyword strategy, bid mgmt, reporting — M

## 2. Resume: redact -> hiring-manager summary

Python equivalent of piping `aibackends task redact-pii` into `aibackends task summarize`.

In [6]:
RESUME_PII_LABELS = [
    "person_name", "email", "phone_number", "address", "date_of_birth", "url",
    "linkedin_profile", "github_profile", "company_name", "job_title", "school",
    "identification_number",
]
RESUME_SYSTEM_PROMPT = "You are an expert resume reviewer helping a hiring manager triage candidates."
RESUME_USER_PROMPT = (
    "Summarize the redacted resume below for a hiring manager. Focus on:\n"
    "- Years and breadth of experience\n- Core technical and domain skills\n"
    "- Notable achievements or projects\n- Education highlights\n"
    "- Overall seniority and likely role fit\n\n"
    "Keep it under 200 words. Do not invent details for fields that have been redacted "
    "(they will appear as bracketed placeholders such as [person_name_1])."
)
SUMMARY_STEP = LLMTextGenerator(
    task_name="resume-summarize",
    system_prompt=RESUME_SYSTEM_PROMPT,
    prompt=RESUME_USER_PROMPT,
    input_key="text",
    output_key="summary",
)


class ResumeRedactSummarizer(Pipeline):
    steps = [FileIngestor(), PIIRedactor(backend="gliner", labels=RESUME_PII_LABELS), SUMMARY_STEP]


resume = ResumeRedactSummarizer(**LLM).run(fetch("pdf/resume-sample.pdf"))
print(f"Redacted {len(resume['pii_redaction'].entities_found)} entities\n")
print(resume["text"][:300], "...\n")
print(resume["summary"])

Redacted 11 entities

[PERSON_NAME_1]
[JOB_TITLE_2]
[EMAIL_3] | [PHONE_NUMBER_4] | San Francisco, CA | [LINKEDIN_PROFILE_5]
PROFESSIONAL SUMMARY
Results-driven software engineer with 9+ years of experience designing and deploying scalable backend systems,
AI/ML pipelines, and cloud-native applications. Proven track recor ...

This candidate is a highly experienced Software Engineer with **9+ years** of experience, specializing in designing and deploying scalable backend systems, AI/ML pipelines, and cloud-native applications across Fintech, Proptech, and SaaS domains.

**Core Skills:**
*   **Technical:** Strong proficiency in Python, TypeScript, Go, and modern frameworks (FastAPI, NestJS, React). Deep expertise in AI/ML (LightGBM, PyTorch, LLMs, Hugging Face), database technologies (PostgreSQL, pgvector), and cloud infrastructure (AWS, Terraform, Docker).
*   **Domain:** Proven ability to deliver high-throughput APIs, real-time analytics platforms, and machine learning solutions for fr

## 3. Resume: redact -> summarize -> match to open roles

`TaskRunner` drops any built-in task into a pipeline - here `ClassifyTask` with label descriptions.

In [7]:
from aibackends.schemas.pii import Classification
from aibackends.tasks import ClassifyTask

JOB_OPENINGS = {
    "senior-backend-engineer": "5+ years building distributed backend systems in Python or Go.",
    "ml-engineer": "Model training, evaluation, and shipping ML features with PyTorch and MLOps.",
    "platform-engineer": "Kubernetes, CI/CD, observability, and developer-platform tooling.",
    "data-engineer": "ETL/ELT pipelines, warehousing, streaming; Airflow, dbt, Spark, or Kafka.",
    "engineering-manager": "5+ years engineering plus 2+ years people management.",
}


class ResumeMatchResult(BaseModel):
    summary: str
    role_match: Classification
    pii: PIISummary


class ResumeRoleMatcher(Pipeline):
    steps = [
        FileIngestor(),
        PIIRedactor(backend="gliner", labels=RESUME_PII_LABELS),
        SUMMARY_STEP,
        TaskRunner(
            task=ClassifyTask,
            input_key="text",
            output_key="role_match",
            task_config={
                "labels": list(JOB_OPENINGS),
                "label_descriptions": JOB_OPENINGS,
                "system_prompt": "You are a technical recruiter matching candidates to open roles.",
                "prompt": (
                    "Pick the single best-fitting role for the candidate based on the redacted "
                    "resume below. Base your judgement only on evidence in the resume. "
                    'Respond with JSON keys "label", "confidence", and "all_scores".'
                ),
            },
        ),
    ]


raw = ResumeRoleMatcher(**LLM).run(fetch("pdf/resume-sample.pdf"))
match = ResumeMatchResult(
    summary=raw["summary"], role_match=raw["role_match"], pii=pii_summary(raw["pii_redaction"])
)
print(match.model_dump_json(indent=2))

{
  "summary": "This candidate is a highly experienced Software Engineer with **9+ years** of experience, specializing in designing and deploying scalable backend systems, AI/ML pipelines, and cloud-native applications across Fintech, Proptech, and SaaS domains.\n\n**Core Skills:**\n*   **Technical:** Strong proficiency in Python, TypeScript, Go, and modern frameworks (FastAPI, NestJS, React). Deep expertise in AI/ML (LightGBM, PyTorch, LLMs, Hugging Face) and cloud infrastructure (AWS, Terraform, Docker).\n*   **Domain:** Proven ability to deliver high-impact products in complex sectors like Fintech and Proptech.\n\n**Notable Achievements:**\n*   Architected a real-time video analytics platform processing 2M+ daily data points.\n*   Led a team to deploy an ML recommendation engine that boosted user engagement by 34%.\n*   Reduced manual review time by 70% by deploying a multimodal AI pipeline (VLM + LLM).\n*   Improved CI/CD efficiency by cutting deployment time from 2 hours to under 